In [12]:
from dotenv import load_dotenv
import os
import requests
import pandas as pd
import time

load_dotenv()

api_key = os.getenv("RIOT_API_KEY")
print(api_key)

# Get PUUID from gameName and tagLine
def get_puuid(gameName=None, tagLine=None, region=None, api_key=None):
    if gameName is None or tagLine is None:
        raise ValueError("gameName and tagLine must be provided")
    
    root_url = f"https://{region}.api.riotgames.com/"
    url = f"{root_url}riot/account/v1/accounts/by-riot-id/{gameName}/{tagLine}?api_key={api_key}"
    
    response = requests.get(url)
    
    if response.status_code == 200:
        data = response.json()['puuid']
        return data
    else:
        print(f"Error: {response.status_code} - {response.text}")
        return None

# Get gameName and tagLine from PUUID
def get_name_and_tag(puuid=None, region=None, api_key=None):
    if puuid is None:
        raise ValueError("puuid must be provided")
    
    root_url = f"https://{region}.api.riotgames.com/"
    url = f"{root_url}riot/account/v1/accounts/by-puuid/{puuid}?api_key={api_key}"
    
    response = requests.get(url)
    
    if response.status_code == 200:
        data = response.json()
        return data['gameName'], data['tagLine']
    else:
        print(f"Error: {response.status_code} - {response.text}")
        return None, None

# Get ladder data for a specific region and league
def get_ladder(region=None, league=None, api_key=None):
    if region is None:
        raise ValueError("region must be provided")
    
    root_url = f"https://{region}.api.riotgames.com/"
    url = f"{root_url}lol/league/v4/{league}leagues/by-queue/RANKED_SOLO_5x5?api_key={api_key}"
    
    response = requests.get(url)
    
    if response.status_code == 200:
        data = response.json()
        return data
    else:
        print(f"Error: {response.status_code} - {response.text}")
        return None

# Get match history for a specific PUUID
def get_match_history(puuid=None, region=None, count = None, api_key=None):
    if puuid is None:
        raise ValueError("puuid must be provided")
    
    root_url = f"https://{region}.api.riotgames.com/"
    url = f"{root_url}lol/match/v5/matches/by-puuid/{puuid}/ids?count={count}&api_key={api_key}"
    
    response = requests.get(url)
    
    if response.status_code == 200:
        data = response.json()
        return data
    else:
        print(f"Error: {response.status_code} - {response.text}")
        return None

# Get match data for a specific match ID
def get_match_data_from_id(match_id=None, region=None, api_key=None):
    if match_id is None:
        raise ValueError("match_id must be provided")
    
    root_url = f"https://{region}.api.riotgames.com/"
    url = f"{root_url}lol/match/v5/matches/{match_id}?api_key={api_key}"
    
    response = requests.get(url)
    
    if response.status_code == 200:
        data = response.json()
        return data
    else:
        print(f"Error: {response.status_code} - {response.text}")
        return None

# Get match timeline for a specific match ID
def get_match_timeline(match_id=None, region=None, api_key=None):
    if match_id is None:
        raise ValueError("match_id must be provided")
    
    root_url = f"https://{region}.api.riotgames.com/"
    url = f"{root_url}lol/match/v5/matches/{match_id}/timeline?api_key={api_key}"
    
    response = requests.get(url)
    
    if response.status_code == 200:
        return response.json()
    else:
        print(f"Error: {response.status_code} - {response.text}")
        return None

# Process match data into a DataFrame
def process_match_to_df(match_data):
    if not match_data:
        return None

    # Metadata & Info
    metadata = match_data.get('metadata', {})
    info = match_data.get('info', {})
    
    match_id = metadata.get('matchId')
    all_participants_puuids = metadata.get('participants', [])
    
    # Stats
    base_info = {
        'matchId': match_id,
        'all_participants': str(all_participants_puuids), 
        'gameCreation': info.get('gameCreation'),
        'gameDuration': info.get('gameDuration'),
        'gameStartTimeStamp': info.get('gameStartTimestamp'),
        'gameEndTimeStamp': info.get('gameEndTimestamp'),
        'PATCH': info.get('gameVersion') # Current patch
    }

    # Bans és Objectives
    teams_stats = {}
    for team in info.get('teams', []):
        t_id = team.get('teamId')
        
        # Bans
        team_bans = [ban.get('championId') for ban in team.get('bans', [])]
        
        # Objectives
        objs = team.get('objectives', {})
        obj_data = {
            'team_baron_kills': objs.get('baron', {}).get('kills'),
            'team_dragon_kills': objs.get('dragon', {}).get('kills'),
            'team_horde_kills': objs.get('horde', {}).get('kills'),
            'team_inhibitor_kills': objs.get('inhibitor', {}).get('kills'),
            'team_riftHerald_kills': objs.get('riftHerald', {}).get('kills'),
            'team_tower_kills': objs.get('tower', {}).get('kills')
        }
        
        teams_stats[t_id] = {
            'bans': team_bans,
            **obj_data
        }

    # ParticipantDto
    rows = []
    for p in info.get('participants', []):
        p_stats = {
            'assists': p.get('assists'),
            'champLevel': p.get('champLevel'),
            'championId': p.get('championId'),
            'deaths': p.get('deaths'),
            'detectorWardsPlaced': p.get('detectorWardsPlaced'),
            'damageDealtToObjectives': p.get('damageDealtToObjectives'),
            'damageSelfMitigated': p.get('damageSelfMitigated'),
            'firstBloodKill': p.get('firstBloodKill'),
            'firstTowerKill': p.get('firstTowerKill'),
            'gameEndedInEarlySurrender': p.get('gameEndedInEarlySurrender'),
            'gameEndedInSurrender': p.get('gameEndedInSurrender'),
            'goldEarned': p.get('goldEarned'),
            'item0': p.get('item0'),
            'item1': p.get('item1'),
            'item2': p.get('item2'),
            'item3': p.get('item3'),
            'item4': p.get('item4'),
            'item5': p.get('item5'),
            'item6': p.get('item6'),
            'kills': p.get('kills'),
            'lane': p.get('lane'),
            'neutralMinionsKilled': p.get('neutralMinionsKilled'),
            'objectivesStolen': p.get('objectivesStolen'),
            'participantId': p.get('participantId'),
            'totalDamageDealtToChampions': p.get('totalDamageDealtToChampions'),
            'totalDamageTaken': p.get('totalDamageTaken'),
            'totalDamageShieldedOnTeammates': p.get('totalDamageShieldedOnTeammates'),
            'visionScore': p.get('visionScore'),
            'win': p.get('win'),
            'teamId': p.get('teamId'),
            'riotIdGameName': p.get('riotIdGameName')
        }
        
        combined_row = {**base_info, **p_stats, **teams_stats.get(p.get('teamId'), {})}
        rows.append(combined_row)

    return pd.DataFrame(rows)

# Get player-specific stats from match data
def get_player_stats_from_match(match_data, target_puuid):
    if not match_data:
        return None

    # Metadata & Info
    metadata = match_data.get('metadata', {})
    info = match_data.get('info', {})
    
    p = next((player for player in info.get('participants', []) if player.get('puuid') == target_puuid), None)
    
    if not p:
        print(f"A játékos ({target_puuid}) nem szerepelt ebben a meccsben.")
        return None

    player_stats = {
        # Match - MetadataDto
        'matchId': metadata.get('matchId'),
        'participants': metadata.get('participants'),
        
        # Match - InfoDto
        'gameCreation': info.get('gameCreation'),
        'gameDuration': info.get('gameDuration'),
        'gameStartTimeStamp': info.get('gameStartTimestamp'),
        'gameEndTimeStamp': info.get('gameEndTimestamp'),
        'PATCH': info.get('gameVersion'),
        
        # ParticipantDto
        'assists': p.get('assists'),
        'champLevel': p.get('champLevel'),
        'championId': p.get('championId'),
        'deaths': p.get('deaths'),
        'detectorWardsPlaced': p.get('detectorWardsPlaced'),
        'damageDealtToObjectives': p.get('damageDealtToObjectives'),
        'damageSelfMitigated': p.get('damageSelfMitigated'),
        'firstBloodKill': p.get('firstBloodKill'),
        'firstTowerKill': p.get('firstTowerKill'),
        'gameEndedInEarlySurrender': p.get('gameEndedInEarlySurrender'),
        'gameEndedInSurrender': p.get('gameEndedInSurrender'),
        'goldEarned': p.get('goldEarned'),
        'item0': p.get('item0'), 'item1': p.get('item1'), 'item2': p.get('item2'),
        'item3': p.get('item3'), 'item4': p.get('item4'), 'item5': p.get('item5'),
        'item6': p.get('item6'),
        'kills': p.get('kills'),
        'lane': p.get('lane'),
        'neutralMinionsKilled': p.get('neutralMinionsKilled'),
        'objectivesStolen': p.get('objectivesStolen'),
        'participantId': p.get('participantId'),
        'totalDamageDealtToChampions': p.get('totalDamageDealtToChampions'),
        'totalDamageTaken': p.get('totalDamageTaken'),
        'totalDamageShieldedOnTeammates': p.get('totalDamageShieldedOnTeammates'),
        'visionScore': p.get('visionScore'),
        'win': p.get('win'),
        'riotIdGameName': p.get('riotIdGameName')
    }

    # Bans & Objectives
    player_team_id = p.get('teamId')
    team_data = next((t for t in info.get('teams', []) if t.get('teamId') == player_team_id), {})
    
    # BanDto
    player_stats['bans'] = [ban.get('championId') for ban in team_data.get('bans', [])]
    
    # ObjectivesDto
    objs = team_data.get('objectives', {})
    player_stats['team_baron_kills'] = objs.get('baron', {}).get('kills')
    player_stats['team_dragon_kills'] = objs.get('dragon', {}).get('kills')
    player_stats['team_horde_kills'] = objs.get('horde', {}).get('kills')
    player_stats['team_inhibitor_kills'] = objs.get('inhibitor', {}).get('kills')
    player_stats['team_riftHerald_kills'] = objs.get('riftHerald', {}).get('kills')
    player_stats['team_tower_kills'] = objs.get('tower', {}).get('kills')

    return player_stats

# Process timeline data into a DataFrame
def process_timeline_to_frames(timeline_data):
    match_id = timeline_data['metadata']['matchId']
    rows = []
    
    # Frame-ek feldolgozása
    for frame_idx, frame in enumerate(timeline_data['info']['frames']):
        timestamp = frame['timestamp']
        
        # Current data
        for p_id, p_info in frame['participantFrames'].items():
            row = {
                'matchId': match_id,
                'minute': frame_idx,
                'timestamp': timestamp,
                'participantId': int(p_id),
                'totalGold': p_info.get('totalGold'),
                'currentGold': p_info.get('currentGold'),
                'xp': p_info.get('xp'),
                'level': p_info.get('level'),
                'minionsKilled': p_info.get('minionsKilled'),
                'jungleMinionsKilled': p_info.get('jungleMinionsKilled'),
                'positionX': p_info.get('position', {}).get('x'),
                'positionY': p_info.get('position', {}).get('y')
            }
            rows.append(row)
            
    return pd.DataFrame(rows)

# Process timeline events into a DataFrame
def process_timeline_events(timeline_data):
    match_id = timeline_data['metadata']['matchId']
    events_list = []
    
    for frame in timeline_data['info']['frames']:
        for event in frame.get('events', []):
            if event['type'] in ['CHAMPION_KILL', 'ELITE_ monster_kill', 'BUILDING_KILL']: # Csak a fontos eventek
                event_row = {
                    'matchId': match_id,
                    'timestamp': event.get('timestamp'),
                    'type': event.get('type'),
                    'killerId': event.get('killerId'),
                    'victimId': event.get('victimId'),
                    'monsterType': event.get('monsterType'),
                    'teamId': event.get('teamId'),
                    'positionX': event.get('position', {}).get('x'),
                    'positionY': event.get('position', {}).get('y')
                }
                events_list.append(event_row)
                
    return pd.DataFrame(events_list)

RGAPI-2e27a924-80d5-487c-9773-7c6003c4daec


In [2]:
player = get_puuid(gameName="Peyz", tagLine="KR11", region="asia", api_key=api_key)
player

'SlQZ2zlvU0VGYNvISzXkK8v6bTF_gFr44fnHfMPZH8QaLZYmMo_XjpJ28nMAlKVBWjtIt3xZUSCA-Q'

In [8]:
# Regions & Leagues
regions = ["kr", "euw1", "na1", "br1", "eun1", "jp1", "la1", "la2", "me1", "oc1", "ru", "sg2", "tr1", "tw2", "vn2"]
leagues = ["challenger", "grandmaster", "master"]

all_ladders = []

# Limit counter
request_count = 0

for region in regions:
    for league in leagues:
        
        # RATE LIMIT
        request_count += 1
        
        # If we hit 100 requests, wait for 125 seconds to respect API limits
        if request_count % 100 == 0:
            print(f"\nLimit reached: {request_count} requests. Waiting for 125 seconds to respect API limits...")
            time.sleep(125)
        # If we hit 20 requests, wait for 2 seconds to avoid hitting short-term limits
        elif request_count % 20 == 0:
            print(f"\n[-] Limit reached: {request_count} requests. Waiting for 2 seconds...")
            time.sleep(2)
        else:
            time.sleep(0.05)

        print(f"Query: {region.upper()} - {league} (Request #{request_count})")
        data = get_ladder(region=region, league=league, api_key=api_key)
        
        if data and 'entries' in data:
            df = pd.DataFrame(data['entries'])
            
            # New columns for region and league
            df['region'] = region
            df['league'] = league
            
            # Append to the list of DataFrames
            all_ladders.append(df)

# Combine all ladders into a single DataFrame
if all_ladders:
    final_ladder_df = pd.concat(all_ladders, ignore_index=True)
    print(f"\nSuccess! {len(final_ladder_df)} players' data loaded.")
else:
    print("\nFailed to retrieve data.")

final_ladder_df.head()

Query: KR - challenger (Request #1)
Query: KR - grandmaster (Request #2)
Query: KR - master (Request #3)
Query: EUW1 - challenger (Request #4)
Query: EUW1 - grandmaster (Request #5)
Query: EUW1 - master (Request #6)
Query: NA1 - challenger (Request #7)
Query: NA1 - grandmaster (Request #8)
Query: NA1 - master (Request #9)
Query: BR1 - challenger (Request #10)
Query: BR1 - grandmaster (Request #11)
Query: BR1 - master (Request #12)
Query: EUN1 - challenger (Request #13)
Query: EUN1 - grandmaster (Request #14)
Query: EUN1 - master (Request #15)
Query: JP1 - challenger (Request #16)
Query: JP1 - grandmaster (Request #17)
Query: JP1 - master (Request #18)
Query: LA1 - challenger (Request #19)

[-] Limit reached: 20 requests. Waiting for 2 seconds...
Query: LA1 - grandmaster (Request #20)
Query: LA1 - master (Request #21)
Query: LA2 - challenger (Request #22)
Query: LA2 - grandmaster (Request #23)
Query: LA2 - master (Request #24)
Query: ME1 - challenger (Request #25)
Query: ME1 - grandmast

,puuid,leaguePoints,rank,wins,losses,veteran,inactive,freshBlood,hotStreak,region,league
0,B3LeP1fhXLpgkaprZU2Ngda93SGOGD7NHMdvE0Izu3u_O1...,2140,I,153,102,True,False,False,False,kr,challenger
1,W8J74uHpVp6Omv3mS1MNW7lmtiptZrTRIG36IF1TpqXChy...,2093,I,302,210,True,False,False,True,kr,challenger
2,3EirdviOjguN0PTsjt88X9SHhMWs7lYVBTwyTcUGSfcZbU...,2092,I,143,95,True,False,False,False,kr,challenger
3,d7MZYeBTxXWrK11_6g5coLD1lalTKF_n8Lv2wZTJ1qQwd0...,2068,I,381,318,True,False,False,True,kr,challenger
4,SlQZ2zlvU0VGYNvISzXkK8v6bTF_gFr44fnHfMPZH8QaLZ...,2067,I,163,112,True,False,False,False,kr,challenger


In [9]:
final_ladder_df.to_csv("final_ladder3.csv", index=False)

In [13]:
try:
    # Read the player data from the CSV file
    players_df = pd.read_csv("final_ladder3.csv")
    
except FileNotFoundError:
    print("Error: final_ladder3.csv not found.")
    exit()

unique_match_ids = set() # No duplicates
request_count = 0

total_players = len(players_df)
print(f"Total players to process: {total_players}.")

# Iterate through each player and get their match history
for index, row in enumerate(players_df.itertuples(), 1):
    puuid = row.puuid
    # Routing region is based on the server region (na1, kr, euw1, etc.)
    server_region = row.region 
    
    # Leképezzük a szervert nagy régióra a Match-V5 API-hoz
    if server_region in ['kr', 'jp1', 'tw2', 'vn2']:
        routing_region = "asia"
    elif server_region in ['euw1', 'eun1', 'tr1', 'ru', 'me1']:
        routing_region = "europe"
    elif server_region in ['na1', 'br1', 'la1', 'la2']:
        routing_region = "americas"
    else:
        routing_region = "sea"

    print(f"[{index}/{total_players}] Querying {puuid[:8]}... ({routing_region})")
    
    # Rate Limit Handling
    request_count += 1
    if request_count % 100 == 0:
        print(f"\n[!] 100 requests reached. Waiting for 125 seconds to respect API limits")
        time.sleep(125)
    elif request_count % 20 == 0:
        time.sleep(2)
    else:
        time.sleep(0.05)
    
    # Maximum 100 matches per request
    matches = get_match_history(puuid=puuid, region=routing_region, count=100, api_key=api_key)
    
    if matches:
        # Update the set of unique match IDs with the new matches
        unique_match_ids.update(matches)
    elif matches is None:
         # Request failed (e.g., Riot server error), it's worth waiting a bit before the next attempt
         time.sleep(5)

matches_df = pd.DataFrame(list(unique_match_ids), columns=['matchId'])

# Save the unique match IDs to a CSV file
matches_df.to_csv("unique_matches.csv", index=False)
print("The unique match IDs have been saved to 'unique_matches.csv'.")

matches_df.head()

Total players to process: 85686.
[1/85686] Querying B3LeP1fh... (asia)
[2/85686] Querying W8J74uHp... (asia)
[3/85686] Querying 3EirdviO... (asia)
[4/85686] Querying d7MZYeBT... (asia)
[5/85686] Querying SlQZ2zlv... (asia)
[6/85686] Querying MYIYDC0k... (asia)
[7/85686] Querying jXPyONsm... (asia)
[8/85686] Querying fDsCcn_B... (asia)
[9/85686] Querying 2axpVf7t... (asia)
[10/85686] Querying gJPa6_T0... (asia)
[11/85686] Querying rhSG1Vf0... (asia)
[12/85686] Querying dMiJ65It... (asia)
[13/85686] Querying Z5P_UkE9... (asia)
[14/85686] Querying 9K5xEdZF... (asia)
[15/85686] Querying UKlUPphO... (asia)
[16/85686] Querying gteLM-hS... (asia)
[17/85686] Querying WeTpft7a... (asia)
[18/85686] Querying BhrrBalc... (asia)
[19/85686] Querying TQtoQNim... (asia)
[20/85686] Querying lwASZkN8... (asia)
[21/85686] Querying MMgJHk7b... (asia)
[22/85686] Querying RGFWpD0F... (asia)
[23/85686] Querying siMR6yL5... (asia)
[24/85686] Querying NMt9Dn_m... (asia)
[25/85686] Querying 9az4FLxu... (asia)
[

KeyboardInterrupt: 

# Ötletek

Korai előny predikció (win prediction)

Win probability percről percre

Gold és XP regression

Térbeli elemzés (koordinátákkal)

Gold management

Tárgyak győzelmi aránya